# Đánh giá 231 câu (evaluator.py + recall_at_k + ablation) — chạy trên Colab (D-163/D-164)

**Bấm Run all rồi đợi.** Notebook này KHÔNG chạy ETL gì cả — DB đã ở trạng thái CUỐI
CÙNG (D-162, `v4_formula_hybrid_fix` / `v19_pill_kernels`, 16.515 chunk), người dùng
đã tự tay upload TOÀN BỘ `database/` hiện có lên Drive (thư mục `database_png`, cùng
cấp `datasource_png`) — **không phải một checkpoint rút gọn kiểu ETL**. Notebook chỉ
đọc DB đó và chạy ba phép đo mà `report/tex_source/` đang cần cập nhật:

1. `recall_at_k.py` — Recall@k thô + rerank, KHÔNG gọi LLM.
2. `ablation.py` — bảng 3 kênh × rerank × cổng lọc (12 cấu hình), KHÔNG gọi LLM.
3. `evaluator.py` — pipeline RAG thật (retrieval → Qwen2.5-3B sinh câu trả lời → LLM
   thứ hai chấm điểm) cho **231 câu / 12 quyển**, chạy TỪNG QUYỂN MỘT + đồng bộ kết
   quả lên Drive ngay sau mỗi quyển (giống triết lý `colab_runtime_etl.ipynb`: một
   quyển lỗi/phiên bị ngắt không mất tiến độ các quyển trước).

## Vì sao chạy trên Colab thay vì máy dev, và vì sao KHÔNG tải zip về

Máy dev có GPU **GTX 1050 Ti (4 GB VRAM)** — đo được bước 3 tốn **~3–3,5 phút/câu**
(hai quyển `SGK_KHTN_6_KNTT` 19 câu + `SGK_KHTN_6_CD` 20 câu chạy xong trên máy dev
với LLM giám khảo Groq mới, kết quả VẪN DÙNG ĐƯỢC nhưng notebook này cố ý chạy lại cả
12 quyển để 231 câu ra từ MỘT môi trường duy nhất, xem D-164). 231 câu ước tính
5–12 giờ trên máy dev; Colab Pro (GPU tốt hơn) nên nhanh hơn nhiều cho phần sinh câu
trả lời Qwen2.5-3B — đó là khâu chiếm hầu hết thời gian, KHÔNG phải bước gọi Groq
(Groq trả lời dưới 1 giây/lượt, đo D-163).

**Mọi kết quả (báo cáo tổng hợp + `*_result.csv` từng quyển) được đồng bộ THẲNG lên
Drive** (`EVAL_RESULTS_DRIVE_DIR` ở mục 6), không dùng `files.download()` — đúng yêu
cầu "mọi kết quả lưu trên Drive hết". Cell cuối chỉ in đường dẫn Drive, không tải gì
về máy qua trình duyệt.

## LLM giám khảo: Groq, HAI model xoay vòng (D-163)

`stealth/ox-alpha` (OpenRouter) đã hết free. Nay dùng Groq:
`qwen/qwen3.8-27b` + `openai/gpt-oss-120b`, xoay vòng qua `JudgePool`
(`src/test/eval_llm.py`) khi một model bị rate-limit (đo được: 8000 token/phút/model,
RIÊNG từng model). **Khoá Groq lấy từ Colab Secrets, KHÔNG gõ thẳng vào notebook**
(notebook này được commit vào git).

## 1. Clone repo

In [ ]:
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git
%cd project-bio-rag

In [ ]:
!git log --oneline -3

## 2. Cài dependencies

Không cần `mineru_vl_utils`/pin `transformers` như notebook ETL — đường đánh giá
không gọi MinerU (bước OCR công thức chỉ chạy lúc ETL, đã xong). Cũng không cần
`poppler`/`tesseract` (không OCR trang nào ở đây).

In [ ]:
!pip install -q -r requirements.txt
import transformers
print("transformers:", transformers.__version__)

## 3. Secrets (đặt TRƯỚC khi tải model)

Mở tab 🔑 (Secrets) bên trái, thêm HAI khoá, bật *Notebook access* cho cả hai:

- `HF_TOKEN` — tải model từ HuggingFace (Qwen2.5-3B-Instruct, bge-m3, reranker, CLIP).
- `GROQ_API_KEY` — LLM giám khảo (D-163). **Không dán khoá thẳng vào ô code** — notebook
  này commit vào git công khai được, một khoá lộ trong đó là khoá phải thu hồi.

In [ ]:
import os, multiprocessing
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)
os.environ["USE_GPU"] = "true"
print("OK, cpu count:", n)

## 4. Tải model về `./models` (chạy ONLINE, trước khi bật offline)

Dùng profile `serve` (`src/utils/download_models.py`) — đúng bốn model cần cho
truy vấn + sinh câu trả lời: `bge-m3` (embedding), `bge-reranker-v2-m3` (rerank),
`Qwen2.5-3B-Instruct` (sinh câu trả lời), `clip-vit-base-patch16` (ảnh — AppServices
nạp cả collection ảnh dù bộ test này chỉ có câu hỏi văn bản/đã có gold key text).

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "-u", "./src/utils/download_models.py",
                    "--save_dir", "./models", "--profile", "serve"])
if r.returncode != 0:
    raise RuntimeError(
        f"Tai model that bai (ma thoat {r.returncode}) - DUNG o day.")
print("Tai model xong, ma thoat 0.")

## 5. Mount Drive + đường dẫn

**KHÔNG có checkpoint kiểu ETL ở đây** — người dùng đã tự tay upload TOÀN BỘ
`database/` hiện có lên Drive, đặt tên `database_png` (song song `datasource_png`
của notebook ETL). Trong `database/images/<quyển>/`, thư mục con `snapshot/` đã bị
XOÁ TAY để giảm dung lượng — chỉ còn hình đã cắt; không ảnh hưởng bước đánh giá này
(evaluator.py không đọc file ảnh, chỉ đọc `biology_text`/BM25/`processing_status`).

**Sửa `DB_SOURCE_DIR` dưới đây nếu tên/đường dẫn thư mục trên Drive của bạn khác** —
đây là suy đoán theo quy ước đặt tên `datasource_png`, CHƯA được xác minh trực tiếp
trên Drive thật của bạn.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

DB_SOURCE_DIR = "/content/drive/MyDrive/project_bio_rag/database_png"  # SUA neu khac
os.environ["RAG_DATABASE_DIR"] = "/content/database"  # dia cuc bo, tranh Drive-FUSE cho SQLite (D-152)
EVAL_RESULTS_DRIVE_DIR = "/content/drive/MyDrive/project_bio_rag/eval_240_results"

base = "/content/project-bio-rag/models"
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["RERANK_MODEL"] = f"{base}/bge-reranker-v2-m3"
os.environ["LLM_MODEL"] = f"{base}/Qwen2.5-3B-Instruct"
os.environ["CLIP_MODEL"] = f"{base}/clip-vit-base-patch16"
os.environ["HF_HUB_OFFLINE"] = "1"  # model da tai o muc 4

for key in ("RAG_DATABASE_DIR", "EMBEDDING_MODEL", "RERANK_MODEL", "LLM_MODEL",
            "CLIP_MODEL", "HF_HUB_OFFLINE"):
    print(f"{key} = {os.environ[key]}")
print("DB_SOURCE_DIR =", DB_SOURCE_DIR)
print("EVAL_RESULTS_DRIVE_DIR =", EVAL_RESULTS_DRIVE_DIR)

## 6. Hàm copy chịu Drive-FUSE rớt kết nối (giống `colab_runtime_etl.ipynb`, D-161)

Dùng cho CẢ HAI hướng: đọc `database_png` xuống đĩa cục bộ, và ghi kết quả lên
`EVAL_RESULTS_DRIVE_DIR`.

In [ ]:
import shutil
import time
from pathlib import Path


def _copy_resilient(src: Path, dst: Path, tries: int = 5, delay: float = 5.0):
    """Di tung file, thu lai khi Drive-FUSE rot ket noi (ENOTCONN, D-161) - mot
    file loi khong huy phan cay da sao chep duoc."""
    if src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for child in src.iterdir():
            _copy_resilient(child, dst / child.name, tries, delay)
        return
    for attempt in range(1, tries + 1):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as exc:
            if attempt == tries:
                raise
            print(f"  loi copy {src.name} (lan {attempt}/{tries}): {exc} -> thu lai sau {delay}s")
            time.sleep(delay)


print("OK, dinh nghia xong _copy_resilient")

## 7. Khôi phục DB (CHỈ ĐỌC từ `database_png`)

Copy TOÀN BỘ `DB_SOURCE_DIR` vào đĩa cục bộ — không có logic loại trừ/checkpoint
từng phần như ETL, vì đây là bản upload MỘT LẦN, đã ở trạng thái cuối.

In [ ]:
src_dir = Path(DB_SOURCE_DIR)
if not src_dir.exists():
    parent = src_dir.parent
    goi_y = list(parent.iterdir()) if parent.exists() else []
    raise RuntimeError(
        f"Khong thay {src_dir}. Cac thu muc con trong {parent}:\n"
        + "\n".join(str(p) for p in goi_y)
        + "\n-> sua DB_SOURCE_DIR o muc 5 cho dung, dung doan.")

local_db = Path(os.environ["RAG_DATABASE_DIR"])
local_db.mkdir(parents=True, exist_ok=True)
for item in src_dir.iterdir():
    print("dang khoi phuc:", item.name)
    _copy_resilient(item, local_db / item.name)
print("XONG khoi phuc DB tu Drive.")

## 8. Xác nhận index khôi phục đúng — ĐỪNG bỏ qua bước này

Đo trực tiếp trên chính DB vừa khôi phục, không tin tên thư mục Drive. Kỳ vọng
(khớp D-162, đo trên máy dev cùng ngày): `processing_status` 2399/2399 trang ở
`text_extraction_version = v4_formula_hybrid_fix`; `biology_text` **16.515** chunk;
BM25 (`database/sparse/bm25_meta.json`) **16.515** id.

In [ ]:
import json
import sys
from collections import Counter

sys.path.insert(0, "/content/project-bio-rag")
import chromadb

client = chromadb.PersistentClient(path=os.environ["RAG_DATABASE_DIR"])

ps = client.get_collection("processing_status")
docs = [json.loads(d) for d in ps.get(include=["documents"])["documents"]]
print("processing_status:", len(docs))
print("text_extraction_version:", Counter(d.get("text_extraction_version") for d in docs))

bt = client.get_collection("biology_text")
print("biology_text count:", bt.count())

bm25_meta_path = Path(os.environ["RAG_DATABASE_DIR"]) / "sparse" / "bm25_meta.json"
if bm25_meta_path.exists():
    with open(bm25_meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    print("BM25 ids:", len(meta["ids"]), "| vocab:", len(meta["vocab"]))
else:
    print("CANH BAO: khong thay", bm25_meta_path, "- BM25/hybrid se khong chay duoc.")

assert bt.count() == 16515, f"So chunk khong khop ky vong D-162 (16515), do duoc {bt.count()} - DUNG, kiem tra lai DB_SOURCE_DIR."
print("\nOK - index khop ky vong D-162, chay tiep duoc.")

## 9. Cấu hình LLM giám khảo (Groq, D-163)

In [ ]:
os.environ["EVAL_LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["EVAL_LLM_API_KEY"] = os.environ["GROQ_API_KEY"]
os.environ["EVAL_LLM_MODEL"] = "qwen/qwen3.8-27b"
os.environ["EVAL_LLM_MODELS"] = "qwen/qwen3.8-27b,openai/gpt-oss-120b"

from src.test.eval_llm import get_eval_llm, is_configured
print("configured:", is_configured())
_llm = get_eval_llm(temperature=0.0)
_resp = _llm.invoke("Tra loi dung mot chu: OK")
print("smoke test:", _resp.content)

## 10. Hàm đồng bộ kết quả lên Drive

Gọi sau MỖI bước/MỖI quyển — chỉ đồng bộ file kết quả (nhỏ), không đụng `database/`.
Nếu `EVAL_RESULTS_DRIVE_DIR` đã có dữ liệu từ một phiên trước bị ngắt, khôi phục
NGƯỢC lại vào repo local trước (resume), rồi mới bắt đầu vòng lặp — quyển nào đã có
`*_result.csv` khôi phục được sẽ bị `--bo-qua-da-co` bỏ qua, không tính lại.

In [ ]:
OUT_FILES_TOP = ["evaluation_report_240.csv", "evaluation_report_240.md",
                 "recall_at_k_report.csv", "recall_at_k_report.md",
                 "ablation_report_240.csv", "ablation_report_240.md"]

drive_out = Path(EVAL_RESULTS_DRIVE_DIR)


def dong_bo_ket_qua():
    drive_out.mkdir(parents=True, exist_ok=True)
    (drive_out / "testsets_240").mkdir(parents=True, exist_ok=True)
    for name in OUT_FILES_TOP:
        p = Path("src/test") / name
        if p.exists():
            _copy_resilient(p, drive_out / name)
    for p in Path("src/test/testsets_240").glob("*_result.csv"):
        _copy_resilient(p, drive_out / "testsets_240" / p.name)
    print("Da dong bo ket qua ->", drive_out)


def khoi_phuc_tien_do_da_co():
    """Resume: keo ket qua tu phien Colab TRUOC (neu co) ve lai repo local."""
    if not drive_out.exists():
        print("Chua co ket qua nao tren Drive - bat dau lot moi.")
        return
    for name in OUT_FILES_TOP:
        p = drive_out / name
        if p.exists():
            _copy_resilient(p, Path("src/test") / name)
    src_results = drive_out / "testsets_240"
    n = 0
    if src_results.exists():
        for p in src_results.glob("*_result.csv"):
            _copy_resilient(p, Path("src/test/testsets_240") / p.name)
            n += 1
    print(f"Da khoi phuc {n} file *_result.csv tu lan chay truoc (neu co).")


khoi_phuc_tien_do_da_co()

## 11. Bước 1 — `recall_at_k.py` (không LLM, nhanh)

In [ ]:
r = subprocess.run([sys.executable, "-u", "src/test/recall_at_k.py",
                    "--testset-dir", "src/test/testsets_240"])
print("exit code:", r.returncode)
dong_bo_ket_qua()

## 12. Bước 2 — `ablation.py` (12 cấu hình, không LLM)

Cache tự vô hiệu theo dấu vân index (digest chunk id + `TEXT_EXTRACTION_VERSION`),
nên không cần xoá `ablation_cache.json` tay — index đổi thì tự tính lại.

In [ ]:
r = subprocess.run([sys.executable, "-u", "-m", "src.test.ablation",
                    "--testset-dir", "src/test/testsets_240",
                    "--out", "src/test/ablation_report_240"])
print("exit code:", r.returncode)
dong_bo_ket_qua()

## 13. Giữ phiên sống trước khi chạy bước dài

In [ ]:
%%javascript
function KeepClicking(){
  var btn = document.querySelector("colab-connect-button");
  if (btn) { btn.click(); console.log("Da bam connect luc " + new Date()); }
}
setInterval(KeepClicking, 60000);

## 14. Bước 3 — `evaluator.py` (mục chính, TỐN THỜI GIAN NHẤT)

Chạy TỪNG QUYỂN một (không phải một lệnh 12-quyển gộp) + đồng bộ Drive ngay sau mỗi
quyển — một quyển lỗi hoặc phiên Colab bị ngắt giữa chừng chỉ mất tiến độ của ĐÚNG
quyển đang chạy dở, 11 quyển kia đã an toàn trên Drive. `--bo-qua-da-co` cho phép
chạy lại ô này sau khi phiên bị ngắt: quyển đã khôi phục được ở mục 10 sẽ bị bỏ qua,
không tốn thêm lượt Groq/Qwen.

In [ ]:
BOOKS = ["SGK_KHTN_6_KNTT", "SGK_KHTN_7_KNTT", "SGK_KHTN_8_KNTT", "SGK_KHTN_9_KNTT",
        "SGK_KHTN_6_CTST", "SGK_KHTN_7_CTST", "SGK_KHTN_8_CTST", "SGK_KHTN_9_CTST",
        "SGK_KHTN_6_CD", "SGK_KHTN_7_CD", "SGK_KHTN_8_CD", "SGK_KHTN_9_CD"]

for idx, book in enumerate(BOOKS, start=1):
    print(f"\n=== [{idx}/12] ({100*(idx-1)/12:.0f}%) {book} ===")
    r = subprocess.run([sys.executable, "-u", "src/test/evaluator.py",
                        "--testset-dir", "src/test/testsets_240", "--hau-to", "_240",
                        "--book", book, "--bo-qua-da-co"])
    print(f"{book} exit code: {r.returncode}")
    if r.returncode != 0:
        raise RuntimeError(
            f"{book} THAT BAI (ma {r.returncode}) - DUNG, xem log truoc khi chay lai "
            "(ket qua cac quyen truoc van con tren Drive).")
    dong_bo_ket_qua()
    print(f"--- {idx}/12 ({100*idx/12:.0f}%) da xong, da dong bo Drive ---")

## 15. Xem lại bảng tổng hợp (không tải gì về máy — đã nằm trên Drive)

In [ ]:
import pandas as pd
d = pd.read_csv("src/test/evaluation_report_240.csv")
print(d.to_string())
print()
print("Tong so cau:", int(d["num_questions"].sum()))
print("\nMoi ket qua da nam o:", EVAL_RESULTS_DRIVE_DIR)